# 🔀 Multi-Query Retrieval
## RAG with LangChain + Ollama + ChromaDB

---

## 🧠 What is Multi-Query Retrieval?

Instead of running **one search** with the user's question, Multi-Query Retrieval:
1. Uses the LLM to generate **4 different phrasings** of the same question
2. Runs a **separate retrieval** for each phrasing
3. **Merges and deduplicates** all retrieved chunks
4. Feeds the richer combined context to the LLM for the final answer

**The Core Problem it Solves:**  
A single query only finds chunks that are *semantically close to that one phrasing*.  
Relevant information scattered across your document under different terminology gets missed.

```
Single Query:  "leave policy"  →  finds chunks mentioning "leave"
                                   misses chunks saying "time off", "absence", "holiday entitlement"

Multi-Query:   "leave policy"           →  chunk A, chunk B
               "time off rules"         →  chunk B, chunk C
               "employee absence"       →  chunk C, chunk D
               "holiday entitlement"    →  chunk D, chunk E
               After dedup:  A, B, C, D, E  →  much richer context
```

**Analogy:**  
Imagine researching a topic in a library. A novice searches once under one keyword and leaves.  
An expert researcher tries *multiple subject headings* — synonyms, related terms, different angles — then combines everything they find.  
Multi-Query Retrieval makes your RAG system the expert researcher.

---

## 🗺️ Full Pipeline Overview

```
┌─────────────────────────────────────────────────────────────┐
│                   INDEXING PHASE (run once)                 │
│  Raw Text → TextLoader → Chunks → Embeddings → ChromaDB    │
└─────────────────────────────────────────────────────────────┘
              ↓  (query time — the Multi-Query flow)
┌─────────────────────────────────────────────────────────────┐
│                   QUERYING PHASE (per question)             │
│                                                             │
│  User Question                                              │
│         │                                                   │
│  [Step 7] Multi-Query Chain (LLM)                           │
│         │  → Generates 4 alternative phrasings             │
│         │                                                   │
│  [Step 8a] Retrieve with Query 1  →  docs [ ]              │
│  [Step 8b] Retrieve with Query 2  →  docs [ ]              │
│  [Step 8c] Retrieve with Query 3  →  docs [ ]  ─→  merge   │
│  [Step 8d] Retrieve with Query 4  →  docs [ ]              │
│         │                                                   │
│  [Step 8e] Deduplicate all retrieved chunks                 │
│         │                                                   │
│  [Step 8f] Build prompt with merged unique context         │
│         │                                                   │
│  [Step 8g] LLM generates grounded answer                   │
└─────────────────────────────────────────────────────────────┘
```

> 💡 **Key Insight:** The deduplication step is not optional — without it, the same chunk appears multiple times in the context, wasting tokens and potentially biasing the LLM toward that chunk.

## 🔄 Progression: Where Multi-Query Fits

| Approach | Search Strategy | Strength |
|---|---|---|
| **Basic RAG** | 1 query → top-K chunks | Simple, fast |
| **Query Rewriting** | 1 rewritten query → top-K chunks | Better term alignment |
| **HyDE** | 1 hypothetical doc → top-K chunks | Best semantic alignment |
| **Multi-Query** ⭐ | N queries → merge → deduplicate | Broadest coverage |

Multi-Query trades **latency** (N retrieval calls) for **recall** (finding more relevant chunks across different phrasings).  
It's the right choice when your document uses inconsistent terminology or when questions can be interpreted multiple ways.

## 📦 Prerequisites & Installation

1. **Ollama running** → https://ollama.com
2. **Models pulled:**
   ```bash
   ollama pull nomic-embed-text
   ollama pull gpt-oss:120b-cloud
   ```
3. **Packages installed:**
   ```bash
   pip install langchain langchain-community langchain-ollama langchain-chroma langchain-core chromadb
   ```
4. **Sample file at** `docs/company_policy.txt`

---
## 📚 Step 0 — Imports

Identical imports to the previous notebooks — the technique changes, the stack stays the same.

| Import | Role |
|---|---|
| `TextLoader` | Reads `.txt` file from disk |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `OllamaEmbeddings` | Converts text → dense vectors |
| `Chroma` | Local vector database |
| `ChatOllama` | Local LLM |
| `PromptTemplate` | Parameterised prompt for the multi-query generator |

In [1]:
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

print("✅ All imports successful")

C:\Users\sr43993\AppData\Local\Temp\ipykernel_11272\1327660044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


✅ All imports successful


---
## 📄 Steps 1–6 — Indexing Pipeline

Identical to the previous notebooks — kept concise here.

> 📖 For deep-dive explanations of each step, refer to the **Basic RAG notebook**.

In [2]:
# ── Step 1: Load Document ────────────────────────────────────────────────────
print("Loading document...\n")

loader = TextLoader("docs/company_policy.txt", encoding="utf-8")
documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview: {documents[0].page_content[:200]}")

Loading document...

✅ Loaded 1 document(s)

📄 Preview: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.


In [3]:
# ── Step 2: Split into Chunks ────────────────────────────────────────────────
print("Splitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1}: {chunk.page_content}")
    print("-" * 50)

Splitting document into chunks...

✅ Total Chunks: 1

🔹 Chunk 1: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.
--------------------------------------------------


In [4]:
# ── Step 3: Embedding Model ──────────────────────────────────────────────────
embeddings = OllamaEmbeddings(model="nomic-embed-text")

test_vec = embeddings.embed_query("test")
print(f"✅ Embedding Model Ready — vector dim: {len(test_vec)}")

✅ Embedding Model Ready — vector dim: 768


In [5]:
# ── Step 4: Create & Persist Vector Store ────────────────────────────────────
print("Creating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"✅ Vector Store Ready — {vectorstore._collection.count()} vectors stored")

Creating Chroma Vector Store...

✅ Vector Store Ready — 6 vectors stored


In [6]:
# ── Step 5: Retriever ────────────────────────────────────────────────────────
# k=2 per query — with 4 queries, we could get up to 8 chunks before dedup
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ Retriever Ready (k=2 per query)")

✅ Retriever Ready (k=2 per query)


In [7]:
# ── Step 6: Load LLM ─────────────────────────────────────────────────────────
print("Loading LLM...\n")

llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

print("✅ LLM Ready")

Loading LLM...

✅ LLM Ready


---
## 🔀 Step 7 — Build the Multi-Query Chain

### What the prompt asks the LLM to do:

```
User asks: "What is the leave policy?"

Multi-Query LLM generates 4 variants:
  1. "annual leave entitlement rules"
  2. "employee time off allowance"
  3. "paid holiday days per year"
  4. "absence and vacation policy guidelines"
```

Each variant will match *different chunks* in ChromaDB — covering vocabulary the original question might have missed.

### Parsing the LLM output — why it matters:

The LLM returns a raw string with one query per line. We must parse this into a clean Python list before looping over it for retrieval.

```python
# Raw LLM output (string):
"- annual leave entitlement rules\n- employee time off allowance\n..."

# After parsing (list):
["annual leave entitlement rules", "employee time off allowance", ...]
```

The `.strip("- ").strip()` removes leading dashes and whitespace that LLMs commonly add to list-style outputs.

> ⚠️ **Edge Case:** LLMs sometimes add numbering (`1.`, `2.`) or extra blank lines. The `if q.strip()` guard filters empty lines. For production, add a regex strip for numbered prefixes too.

> 💡 **Production Note:** LangChain's built-in `MultiQueryRetriever` automates this entire step — but building it manually (as done here) makes the internals transparent and fully customizable.

In [8]:
# ── Multi-Query Prompt ────────────────────────────────────────────────────────
# Instructs the LLM to generate 4 alternative phrasings of the user's question
# "Return ONLY the queries" avoids preamble text that breaks the line parser

multi_query_prompt = PromptTemplate.from_template(
    """
You are an expert search assistant.

Generate 4 different search queries for the user's question.
Each query should represent a different way of asking the same thing —
use synonyms, related terms, and different angles.

Return ONLY the queries. One query per line. No numbering. No explanation.

Question:
{question}
"""
)

# ── Multi-Query Chain (LCEL pipe) ─────────────────────────────────────────────
multi_query_chain = multi_query_prompt | llm

print("✅ Multi-Query Chain Ready")

# ── Sanity Check: See what 4 queries look like for a sample question ──────────
sample_q = "What is the leave policy?"
sample_output = multi_query_chain.invoke({"question": sample_q})

print(f"\n🧪 Sample Multi-Query Generation:")
print(f"\n❓ Original Question: {sample_q}")
print(f"\n📋 Generated Queries (raw LLM output):")
print(sample_output.content.strip())

# ── Parse the raw output into a clean Python list ────────────────────────────
sample_list = [
    q.strip("- ").strip()
    for q in sample_output.content.strip().split("\n")
    if q.strip()   # skip empty lines
]

print(f"\n✅ Parsed into {len(sample_list)} queries:")
for i, q in enumerate(sample_list, 1):
    print(f"  {i}. {q}")

✅ Multi-Query Chain Ready

🧪 Sample Multi-Query Generation:

❓ Original Question: What is the leave policy?

📋 Generated Queries (raw LLM output):
company leave policy details  
employee vacation and sick leave guidelines  
workplace time off policy overview  
HR leave entitlement rules

✅ Parsed into 4 queries:
  1. company leave policy details
  2. employee vacation and sick leave guidelines
  3. workplace time off policy overview
  4. HR leave entitlement rules


---
## 🔑 The Deduplication Step — Why It's Non-Negotiable

With 4 queries each retrieving 2 chunks, you get up to **8 chunks** — but many will overlap.

```
Query 1 retrieved:  [chunk A, chunk B]
Query 2 retrieved:  [chunk B, chunk C]   ← chunk B is a duplicate
Query 3 retrieved:  [chunk C, chunk D]   ← chunk C is a duplicate
Query 4 retrieved:  [chunk D, chunk E]   ← chunk D is a duplicate

Without dedup → context: A B B C C D D E  (8 chunks, lots of repetition)
With dedup    → context: A B C D E        (5 unique chunks, clean context)
```

**Why duplicates are harmful:**
- Waste prompt tokens on repeated content
- Can bias the LLM toward the duplicated chunk (it appears more in context)
- May push other relevant chunks out of the context window

**The dedup strategy used here:**  
A Python `set` tracks `page_content` strings already seen. First occurrence kept, subsequent ones skipped. Preserves original retrieval order (unlike `set()` on the whole list).

---
## 💬 Step 8 — Interactive Q&A Loop with Multi-Query Retrieval

### The Full Flow for Every Question:

```
User question
      │
  [LLM] generates 4 query variants
      │
      ├── Query 1 → ChromaDB → [chunk A, chunk B]
      ├── Query 2 → ChromaDB → [chunk B, chunk C]   ← all_docs grows
      ├── Query 3 → ChromaDB → [chunk C, chunk D]
      └── Query 4 → ChromaDB → [chunk D, chunk E]
                                       │
                               Deduplicate → [A, B, C, D, E]
                                       │
                               Build context string
                                       │
                               LLM generates answer
```

### What to observe while running:
- The 4 generated queries — notice how each covers a different angle
- Which queries retrieve *different* chunks vs overlapping ones
- How the unique doc count compares to the raw retrieved count (dedup effectiveness)
- Whether the final answer covers more ground than a single-query approach would

In [9]:
print("Multi-Query RAG System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:

    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    print("\n===== ❓ ORIGINAL QUESTION =====")
    print(question)

    # ── STEP 8a: GENERATE 4 QUERY VARIANTS ───────────────────────────────────
    # LLM rephrases the question 4 ways to maximise retrieval coverage
    generated_queries = multi_query_chain.invoke({"question": question})
    queries_text = generated_queries.content.strip()

    print("\n===== 🔀 GENERATED QUERIES =====")
    print(queries_text)

    # ── STEP 8b: PARSE INTO A CLEAN LIST ─────────────────────────────────────
    # Strip dashes, whitespace, and blank lines from the LLM's raw output
    query_list = [
        q.strip("- ").strip()
        for q in queries_text.split("\n")
        if q.strip()
    ]

    print("\n===== 📋 QUERY LIST =====")
    for index, query in enumerate(query_list, 1):
        print(f"  {index}. {query}")

    # ── STEP 8c: RETRIEVE FOR EACH QUERY ─────────────────────────────────────
    # Each query independently searches ChromaDB and adds its results to all_docs
    all_docs = []

    for index, query in enumerate(query_list, 1):

        print(f"\n===== 🔍 RETRIEVAL — Query {index}: '{query}' =====")
        retrieved_docs = retriever.invoke(query)

        for doc in retrieved_docs:
            print(f"  → {doc.page_content[:120]}")
            print("  " + "-" * 55)
            all_docs.append(doc)

    print(f"\n  📦 Total raw retrieved (with duplicates): {len(all_docs)} chunks")

    # ── STEP 8d: DEDUPLICATE ──────────────────────────────────────────────────
    # Use a set to track seen content — preserves order, eliminates repeats
    # Without this, the same chunk could appear up to 4 times in the context
    unique_docs = []
    seen = set()

    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)

    print(f"  ✅ After deduplication: {len(unique_docs)} unique chunks")

    print("\n===== 📚 UNIQUE DOCUMENTS (final context) =====")
    for index, doc in enumerate(unique_docs, 1):
        print(f"\n  Unique Chunk {index}: {doc.page_content}")

    # ── STEP 8e: BUILD CONTEXT FROM UNIQUE CHUNKS ────────────────────────────
    # All unique chunks become the LLM's knowledge source for this question
    context = "\n\n".join(
        doc.page_content for doc in unique_docs
    )

    # ── STEP 8f: CRAFT FINAL ANSWER PROMPT ───────────────────────────────────
    # Original question + merged unique context = broadest possible grounded answer
    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

    # ── STEP 8g: GENERATE ANSWER ──────────────────────────────────────────────
    response = llm.invoke(prompt)

    print("\n===== 🤖 FINAL ANSWER =====")
    print(response.content)

Multi-Query RAG System Ready. Type 'exit' to stop.


===== ❓ ORIGINAL QUESTION =====
maternity leave

===== 🔀 GENERATED QUERIES =====
maternity leave policies and duration  
paid maternity leave benefits and eligibility  
how much maternity leave time is allowed in the US  
parental leave options for new mothers

===== 📋 QUERY LIST =====
  1. maternity leave policies and duration
  2. paid maternity leave benefits and eligibility
  3. how much maternity leave time is allowed in the US
  4. parental leave options for new mothers

===== 🔍 RETRIEVAL — Query 1: 'maternity leave policies and duration' =====
  → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

An
  -------------------------------------------------------
  → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

An
  -------------------------------------------------------

===== 🔍 RETRIEVAL — Que

---
## 🧪 Bonus — Single-Shot Multi-Query

Test any question without the interactive loop.

In [10]:
# ── Change this question and re-run ──────────────────────────────────────────
question = "What are my rights if I am terminated?"

# Generate 4 query variants
queries_raw = multi_query_chain.invoke({"question": question}).content.strip()
query_list = [q.strip("- ").strip() for q in queries_raw.split("\n") if q.strip()]

# Retrieve and deduplicate
all_docs = [doc for q in query_list for doc in retriever.invoke(q)]
seen = set()
unique_docs = []
for doc in all_docs:
    if doc.page_content not in seen:
        seen.add(doc.page_content)
        unique_docs.append(doc)

# Build context and answer
context = "\n\n".join(doc.page_content for doc in unique_docs)
prompt = f"""Answer ONLY using the context below.
If not found, say: "I could not find that information in the document."

Context:
{context}

Question: {question}
"""
answer = llm.invoke(prompt).content

print(f"❓ Question        : {question}")
print(f"\n📋 4 Queries used  :")
for i, q in enumerate(query_list, 1):
    print(f"   {i}. {q}")
print(f"\n📦 Raw chunks      : {len(all_docs)} → Unique: {len(unique_docs)}")
print(f"\n🤖 Answer          : {answer}")

❓ Question        : What are my rights if I am terminated?

📋 4 Queries used  :
   1. Employee termination rights and legal protections
   2. What legal rights do I have if I'm fired from my job?
   3. Rights and benefits for workers who are dismissed
   4. What can I claim after being let go from employment?

📦 Raw chunks      : 8 → Unique: 1

🤖 Answer          : I could not find that information in the document.


---
## 🔬 Bonus — Coverage Comparison: Single vs Multi-Query

This cell runs the same question through both approaches and compares how many unique chunks each retrieves.  
The **best demo cell** for showing concretely why Multi-Query retrieves more.

In [11]:
test_questions = [
    "What is the leave policy?",
    "Can I work from home?",
    "What happens if I am late?",
    "What are the performance review criteria?",
]

print(f"{'Question':<45} {'Single-Q Chunks':>16} {'Multi-Q Chunks':>14} {'Extra Coverage':>14}")
print("-" * 92)

for q in test_questions:
    # Single-query retrieval
    single_docs = retriever.invoke(q)
    single_count = len(single_docs)

    # Multi-query retrieval
    queries_raw = multi_query_chain.invoke({"question": q}).content.strip()
    query_list = [x.strip("- ").strip() for x in queries_raw.split("\n") if x.strip()]
    all_docs = [doc for qr in query_list for doc in retriever.invoke(qr)]
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)
    multi_count = len(unique_docs)

    extra = multi_count - single_count
    label = f"+{extra} more" if extra > 0 else "same"

    print(f"{q:<45} {single_count:>16} {multi_count:>14} {label:>14}")

Question                                       Single-Q Chunks Multi-Q Chunks Extra Coverage
--------------------------------------------------------------------------------------------
What is the leave policy?                                    2              1           same
Can I work from home?                                        2              1           same
What happens if I am late?                                   2              1           same
What are the performance review criteria?                    2              1           same


---
## 🎯 Key Takeaways

| Concept | What to Remember |
|---|---|
| **Multi-Query core idea** | Search N times with N phrasings, merge results |
| **Why N queries beat 1** | Different phrasings hit different vocabulary in your documents |
| **Deduplication is mandatory** | Prevents token waste, context bias, and repeated chunks |
| **`seen` set preserves order** | Unlike `list(set(...))` which shuffles — order matters for context quality |
| **k=2 per query** | With 4 queries, you start with up to 8 raw chunks before dedup |
| **LLM called N+1 times** | Once per query generation + once for final answer |

---

## ⚖️ Multi-Query Trade-offs

| ✅ Strengths | ⚠️ Weaknesses |
|---|---|
| Highest recall — finds chunks other methods miss | N retrieval calls = higher latency |
| Works well with inconsistent doc terminology | More LLM calls = higher cost |
| Great for broad or multi-faceted questions | Overkill for simple, specific questions |
| Naturally handles ambiguous phrasings | Dedup logic needed — adds implementation complexity |

---

## 🔄 All Four RAG Approaches — Final Summary

```
Basic RAG:
  question ──────────────────────────────────→ k chunks → answer

Query Rewriting:
  question → LLM rewrites → better query ──→ k chunks → answer

HyDE:
  question → LLM writes hypothetical answer → k chunks → answer
  (answer-shaped search vector)

Multi-Query:
  question → LLM generates 4 variants
             ├── variant 1 → k chunks ─┐
             ├── variant 2 → k chunks  ├── merge → dedup → answer
             ├── variant 3 → k chunks  │
             └── variant 4 → k chunks ─┘
  (broadest coverage)
```

---

## 🚀 Next Steps to Explore

1. **LangChain `MultiQueryRetriever`** — Built-in class that automates Steps 7–8d in one line
2. **Combine with Reranking** — After merging, use a cross-encoder to rerank unique chunks by relevance before building context
3. **Combine with HyDE** — Generate hypothetical docs for each of the 4 queries for maximum retrieval diversity
4. **Dynamic k** — Use a higher `k` for the first query (broad), lower for subsequent ones (focused)
5. **RAGAS evaluation** — Measure context recall improvement of Multi-Query vs Basic RAG across a benchmark question set